# Chronological ML training

Uses generated synthetic prices for software research only; these are not actual market data.

The next-period target uses the following close. The final training row is purged so no training label consumes a validation close. Classification metrics alone do not measure strategy profitability.


In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
while not (repo / "backend").is_dir() and repo != repo.parent:
    repo = repo.parent
sys.path.insert(0, str(repo / "backend"))

import pickle
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from trading.data.synthetic import generate_synthetic_ohlcv
from trading.features.feature_pipeline import engineer_features
from trading.ml.targets import next_period_direction
from trading.ml.trainer import SignalModelTrainer
names = ["return_5", "return_20", "rsi_14", "macd_line", "atr_14",
         "rolling_volatility_20", "volume_change_1"]
frame = engineer_features(generate_synthetic_ohlcv(periods=1200, frequency="1D", seed=17))
frame["target"] = next_period_direction(frame.close)
frame = frame.dropna(subset=[*names, "target"]).reset_index(drop=True)
train_end = int(len(frame) * 0.6)
validation_end = int(len(frame) * 0.8)
train = frame.iloc[:train_end - 1]  # Purge target crossing into validation.
test = frame.iloc[validation_end:]  # Validation remains unused for this baseline.
artifact = SignalModelTrainer().fit_artifact(
    features={name: train[name].astype(float).tolist() for name in names},
    labels=train.target.astype(int).tolist(),
    model_type="logistic_regression")
model = pickle.loads(artifact)  # Only load this trusted, locally generated artifact.
truth = test.target.astype(int).to_numpy()
matrix = test[names].to_numpy(dtype=float)
prediction = model.predict(matrix)
{"accuracy": accuracy_score(truth, prediction),
 "precision": precision_score(truth, prediction, zero_division=0),
 "recall": recall_score(truth, prediction, zero_division=0),
 "f1": f1_score(truth, prediction, zero_division=0)}
